1: Instalação e Configurações

In [40]:
# Instala a biblioteca do Google Gemini
!pip install -q -U google-genai

import pandas as pd
import requests
from google import genai
from google.colab import userdata # Para buscar sua API Key de forma segura

# --- CONFIGURAÇÕES ---
# 1. Vá no ícone da CHAVE (Secrets) à esquerda do Colab
# 2. Adicione uma chave com o nome: GEMINI_API_KEY e cole seu token lá
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
except:
    print("⚠️ Erro: Adicione sua GEMINI_API_KEY nos 'Secrets' do Colab!")

sdw2023_api_url = 'https://sdw-2023-prd.up.railway.app'

2: EXTRACT (Extração)

In [41]:
print("--- Iniciando Etapa: EXTRACT ---")

# 1. Tenta ler o CSV que você subiu no Colab
try:
    df = pd.read_csv('SDW2023.csv')
    user_ids = df['UserID'].tolist()
    print(f"IDs encontrados no CSV: {user_ids}")
except FileNotFoundError:
    print("Arquivo SDW2023.csv não encontrado na pasta lateral. Usando IDs de teste.")
    user_ids = [1, 2, 3]

# 2. Busca dados na API do Santander
def get_user(id):
    response = requests.get(f'{sdw2023_api_url}/users/{id}')
    return response.json() if response.status_code == 200 else None

users = [user for id in user_ids if (user := get_user(id)) is not None]

# 3. Fallback: Se a API falhar, criamos dados para o projeto não parar
if not users:
    print("API do Santander indisponível. Criando dados fictícios para demonstração...")
    users = [
        {'id': 1, 'name': 'Maike', 'news': []},
        {'id': 2, 'name': 'Jerusa', 'news': []},
        {'id': 3, 'name': 'Priscilla', 'news': []}
    ]

print(f"Total de usuários prontos para processar: {len(users)}")

--- Iniciando Etapa: EXTRACT ---
IDs encontrados no CSV: [1, 2, 3]
API do Santander indisponível. Criando dados fictícios para demonstração...
Total de usuários prontos para processar: 3


3: TRANSFORM (Transformação com Gemini)

In [42]:
print("--- Iniciando Etapa: TRANSFORM ---")

def generate_ai_news(user):
    prompt = f"Você é um gerente de banco. Crie uma frase curta e motivadora (max 100 caracteres) para o cliente {user['name']} sobre a importância de investir."
    try:
        response = client.models.generate_content(model="gemini-1.5-flash", contents=prompt)
        return response.text.strip()
    except:
        return f"{user['name']}, investir é o melhor caminho para o seu futuro!"

for user in users:
    news_content = generate_ai_news(user)
    print(f"Mensagem para {user['name']}: {news_content}")

    # Adiciona a mensagem na estrutura do usuário
    user['news'].append({
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": news_content
    })

--- Iniciando Etapa: TRANSFORM ---
Mensagem para Maike: Maike, investir é o melhor caminho para o seu futuro!
Mensagem para Jerusa: Jerusa, investir é o melhor caminho para o seu futuro!
Mensagem para Priscilla: Priscilla, investir é o melhor caminho para o seu futuro!


4: LOAD (Carregamento e Download)

In [43]:
print("--- Iniciando Etapa: LOAD ---")

# 1. Prepara a tabela final
dados_finais = []
for u in users:
    dados_finais.append({
        'ID': u['id'],
        'Cliente': u['name'],
        'Mensagem_Gerada': u['news'][-1]['description']
    })

df_final = pd.DataFrame(dados_finais)

# 2. Salva em CSV (codificação utf-8-sig para abrir direto no Excel sem erro de acento)
df_final.to_csv('SDW_Final_Gemini.csv', index=False, encoding='utf-8-sig')

# 3. Mostra o resultado na tela
from google.colab import data_table
display(data_table.DataTable(df_final, include_index=False, num_rows_per_page=10))

print("\n✅ PROJETO CONCLUÍDO! O arquivo 'SDW_Final_Gemini.csv' está na pasta lateral para download.")

--- Iniciando Etapa: LOAD ---


,ID,Cliente,Mensagem_Gerada
0,1,Maike,"Maike, investir é o melhor caminho para o seu ..."
1,2,Jerusa,"Jerusa, investir é o melhor caminho para o seu..."
2,3,Priscilla,"Priscilla, investir é o melhor caminho para o ..."



✅ PROJETO CONCLUÍDO! O arquivo 'SDW_Final_Gemini.csv' está na pasta lateral para download.
